# 1D J1J2J3 Inference: RNN (Euclidean, Poincare, Lorentz)

This notebook is part of the work arXiv:2604.24337 [quant-ph] (https://arxiv.org/html/2604.24337v1). In this notebook, I performed the inference process for the trained neural quantum states using 10000 samples. The loading directory in the Github repo might differ from the loading path used here. Please check and use the correct loading path. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import time
sys.path.append('utility_lorentz')
from j1j2j3_train_loop_lorentz import *

sys.path.append('utility_poincare')
from j1j2j3_hyprnn_train_loop import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False
GPU Available:  False


In [2]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  
    
def define_load_test(wf,  numsamples,path_to_weights, Ee, clipped_e = False):
    if 'Lorentz' in wf.name:
        test_samples_before = wf.sample_no_tau(numsamples)
    else: 
        test_samples_before = wf.sample(numsamples)
    print(f'The number of samples is {len(test_samples_before)}')
    # --- PART A: Check performance BEFORE loading (Baseline) ---
    wf.model.eval() 
    with torch.no_grad():
        test_gs_before = J1J2J3_local_energies(wf, N, J1, J2, J3, Bz, numsamples, test_samples_before, Marshall_sign=True)
        gs_mean_b = np.round(np.mean(test_gs_before),4)
        gs_var_b = np.round(np.var(test_gs_before),4)
    print(f'Before loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_b}, var E = {gs_var_b}')
    print('====================================================================')

     # --- PART B: Remap and Load the Weights ---
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    #  Load the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        if 'Lorentz' in wf.name:
            test_samples_after = wf.sample_no_tau(numsamples)
        else: 
            test_samples_after = wf.sample(numsamples)        
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2J3_local_energies(wf, N, J1, J2, J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. Apply clipping
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2J3_local_energies(wf, N, J1, J2, J3,Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [3]:
N=100
syssize = 100
numsamples = 10000
units =80
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fcn = 'trained_nqs/J1J2J3'

## J2=0.0, J3=0.5

In [4]:
J2_ = 0.0
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_00_05=-53.9914

In [5]:
# EuclRNN: -37.5197
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/RNN_euclidean/J2={J2_}_J3={J3_}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclRNN_{units}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 6964
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (36.869300842285156-0.0006000000284984708j), var E = 0.5034999847412109
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.51969909667969-0.0003000000142492354j), var E = 0.008500000461935997
DMRG energy  is -53.9914
Time taken=0.119 hrs


In [6]:
# PoincareRNN: -50.6616
rmax=0.78
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/RNN_poincare/J2={J2_}_J3={J3_}_rmax={rmax}/N100_J1=1.0|J2={J2_}|J3={J3_}_HypRNN_{units}_id_hyp_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_00_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 6964
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (36.869300842285156-0.0006000000284984708j), var E = 0.5034999847412109
Successfully remapped and loaded weights.
Clipped 103 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-50.661598205566406-0.001500000013038516j), var E = 7.787300109863281
DMRG energy  is -53.9914
Time taken=0.371 hrs


In [7]:
#LorentzRNN: double clamp: No clipped E
#-53.7407 (with clipped E)
spatial_clamp=4.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=111)
h_wl = f'{fcn}/RNN_lorentz/J2={J2_}_J3={J3_}_sc={spatial_clamp}/N100_J1=1.0|J2={J2_}|J3={J3_}_N=100_LorentzRNN_{units}_ns=80_MsTrue_checkpoint.pt'
#The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (36.89250183105469-9.999999747378752e-05j), var E = 0.5127999782562256
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.73429870605469-0.002400000113993883j), var E = 1.2395000457763672
DMRG energy  is -53.9914
Time taken=0.865 hrs


## J2=0.2, J3=0.2

In [8]:
J2_ = 0.2
J3_ = 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_02=-43.5860

In [9]:
# EuclRNN: -25.3377
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/RNN_euclidean/J2={J2_}_J3={J3_}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclRNN_{units}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 6964
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (34.50659942626953-0.0005000000237487257j), var E = 0.3248000144958496
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-25.33769989013672-0.00039999998989515007j), var E = 0.05609999969601631
DMRG energy  is -43.586
Time taken=0.139 hrs


In [11]:
# PoincareRNN: -41.5420
rmax=0.78
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/RNN_poincare/J2={J2_}_J3={J3_}_rmax={rmax}/N100_J1=1.0|J2={J2_}|J3={J3_}_HypRNN_{units}_id_hyp_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 6964
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (34.50659942626953-0.0005000000237487257j), var E = 0.3248000144958496
Successfully remapped and loaded weights.
Clipped 147 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.54199981689453-0.017799999564886093j), var E = 3.8113999366760254
DMRG energy  is -43.586
Time taken=0.449 hrs


In [12]:
# LorentzRNN: -43.4041
spatial_clamp=4.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=111)
h_wl = f'{fcn}/RNN_lorentz/J2={J2_}_J3={J3_}_sc={spatial_clamp}/N100_J1=1.0|J2={J2_}|J3={J3_}_N=100_LorentzRNN_{units}_ns=80_MsTrue_checkpoint.pt'
#The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (34.525699615478516-9.999999747378752e-05j), var E = 0.33340001106262207
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.40409851074219+0.0017000000225380063j), var E = 0.33570000529289246
DMRG energy  is -43.586
Time taken=1.247 hrs


## J2=0.2, J3=0.5

In [6]:
J2_ = 0.2
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_05=-49.628675088072384

In [14]:
# EuclRNN: -32.5274
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/RNN_euclidean/J2={J2_}_J3={J3_}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclRNN_{units}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 6964
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (41.76750183105469-0.0006000000284984708j), var E = 0.6615999937057495
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-32.527400970458984+0j), var E = 0.004100000020116568
DMRG energy  is -49.6287
Time taken=0.101 hrs


In [7]:
# PoincareRNN: -46.5905
#with clipped e (151 outliers): Mean E = (-46.57780075073242-0.007699999958276749j), var E = 5.74399995803833
rmax=0.78
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/RNN_poincare/J2={J2_}_J3={J3_}_rmax={rmax}/N100_J1=1.0|J2={J2_}|J3={J3_}_HypRNN_{units}_id_hyp_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 6964
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (41.76750183105469-0.0006000000284984708j), var E = 0.6615999937057495
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-46.59049987792969-0.007699999958276749j), var E = 6.245699882507324
DMRG energy  is -49.6287
Time taken=0.623 hrs


In [17]:
#LorentzRNN: -49.1562
spatial_clamp=4.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=111)
h_wl = f'{fcn}/RNN_lorentz/J2={J2_}_J3={J3_}_sc={spatial_clamp}/N100_J1=1.0|J2={J2_}|J3={J3_}_N=100_LorentzRNN_{units}_ns=80_MsTrue_checkpoint.pt'
#The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (41.79410171508789-9.999999747378752e-05j), var E = 0.670799970626831
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-49.15620040893555+0.00930000003427267j), var E = 1.7359999418258667
DMRG energy  is -49.6287
Time taken=1.177 hrs


## J2=0.5, J3=0.2

In [18]:
J2_ = 0.5
J3_= 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_05_02= -38.54733450537742

In [19]:
# EuclRNN: -17.5034
wf = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
e_wl= f'{fcn}/RNN_euclidean/J2={J2_}_J3={J3_}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclRNN_{units}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, e_wl, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 6964
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (41.85390090942383-0.0006000000284984708j), var E = 0.541700005531311
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-17.503400802612305+0.0010000000474974513j), var E = 0.008299999870359898
DMRG energy  is -38.5473
Time taken=0.121 hrs


In [20]:
# PoincareRNN: -36.9487
rmax=0.78
wf = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f'Total params = {total_params}')
h_wl = f'{fcn}/RNN_poincare/J2={J2_}_J3={J3_}_rmax={rmax}/N100_J1=1.0|J2={J2_}|J3={J3_}_HypRNN_{units}_id_hyp_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(wf, numsamples, h_wl, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total params = 6964
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (41.85390090942383-0.0006000000284984708j), var E = 0.541700005531311
Successfully remapped and loaded weights.
Clipped 185 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.948699951171875-0.0003000000142492354j), var E = 2.1451001167297363
DMRG energy  is -38.5473
Time taken=0.579 hrs


In [21]:
#LorentzRNN: -38.3076
spatial_clamp=4.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=111)
h_wl = f'{fcn}/RNN_lorentz/J2={J2_}_J3={J3_}_sc={spatial_clamp}/N100_J1=1.0|J2={J2_}|J3={J3_}_N=100_LorentzRNN_{units}_ns=80_MsTrue_checkpoint.pt'
#The last parameter in LorentzGRU is just the curvature k (empty)
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
t0 = time.time()
define_load_test(wf , numsamples, h_wl, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 6,965
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (41.87810134887695-9.999999747378752e-05j), var E = 0.5544999837875366
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.30759811401367-9.999999747378752e-05j), var E = 0.4837000072002411
DMRG energy  is -38.5473
Time taken=1.392 hrs
